# FPP04L — Shared Cross-Sensor Identity Proxies: CosFace and Proxy-Anchor vs a cosine-BCE control (EfficientNetV2-S, MobileNetV3-Large; 3 repetitions)

**Question:** does adding an **identity-proxy loss** to the pair loss improve verification — above all **cross-sensor (DB1×DB2)** and the **new sensor (FingerDOS)**, where the image model still trails BOZORTH3?

**Shared proxies across sensors.** One learnable 1024-D proxy per **training finger** (60 subjects × 10 fingers = **600 proxies**). A finger's **DB1 and DB2 images share the same proxy** (identity = subject + finger, sensor-free), so the loss pulls both sensors' captures of a finger to one point — explicit cross-sensor invariance. Checked in code: every proxy has images from **both** sensors; validation / calibration / test subjects never get a proxy. **The proxies are discarded after training:** the deployed model is the unchanged encoder (same size, same speed, cosine matching), and the saved run files contain only the encoder.

| Arm | Loss on every batch (64 pairs = 128 images) | Role |
|---|---|---|
| `bce_control` | cosine-BCE (as FPP04 / FPP04C) | same extra fine-tuning **without** proxies: separates the proxy effect from training longer |
| `bce_cosface` | cosine-BCE + λ · **CosFace** over the 600 shared proxies (s = 30, m = 0.35) | margin-softmax representative |
| `bce_proxyanchor` | cosine-BCE + λ · **Proxy-Anchor** over the 600 shared proxies (α = 32, δ = 0.1) | metric-learning representative (few images per class: 8 per finger) |

**Start and training:** every run starts from the **FPP04C full fine-tuned model of the same backbone and repetition** (weights checked); proxies initialised at the L2-normalised **mean embedding of each finger's training images** (both sensors) from that model; full fine-tuning (BatchNorm frozen), AdamW 1e-5, proxies parameterised with norm 0.01 so they move ≈ 100× faster in angle than the encoder weights (no second optimiser); λ = 1; ≤ 50 epochs, patience 10 / LR × 0.3 after 3; batch 64 pairs; FPP01 materialized augmentation (original p 0.25, else one of 3 copies) for training images only.

**FPP protocol (fpp_lib ≥ 1.3):** 60 train / 15 validation / 5 calibration hold-out / 20 test MOLF subjects, FingerDOS external; source-balanced sampler; **early stopping, best epoch and selection on the source-balanced validation EER**; **one frozen threshold = source-balanced validation EER**; bias check with the 5 calibration subjects.

**Reported:** per arm and set (MOLF test balanced, DB1×DB2 test, FingerDOS): AUC, EER, FMR, FNMR, F1-macro, accuracy, precision, recall; **paired differences vs `bce_control` and vs FPP04C** per repetition; standard figures. **Hardware:** A100, High-RAM; 3 arms × 2 backbones × 3 repetitions = 18 runs (≈ 4–5 h; split with `ARMS_TO_RUN`). **Needs:** FPP01, FPP02, **FPP04C finished**, fpp_lib ≥ 1.3.

In [ ]:
%pip install -q openpyxl

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Load the shared libraries from output/00_code/ (fpf_lib from FPF00, fpp_lib installed by FPP01), so every FPP kernel uses the same version.
import shutil, sys, importlib
from pathlib import Path
def _find_folder(name, root=Path('/content/drive/MyDrive'), max_depth=4):
    frontier = [root]
    for _ in range(max_depth + 1):
        nxt = []
        for d in frontier:
            try: kids = [c for c in d.iterdir() if c.is_dir()]
            except OSError: continue
            for c in kids:
                if c.name == name: return c
                nxt.append(c)
        frontier = nxt
    return None
ROOT = _find_folder('FingerPrint-Final')
if ROOT is None: raise FileNotFoundError("'FingerPrint-Final' not found under My Drive")
CODE = ROOT / 'output' / '00_code'
for m in ('fpf_lib.py', 'fpf_tf.py', 'fpp_lib.py'):
    src = CODE / m
    if not src.is_file(): raise FileNotFoundError(f'{src} missing: run ' + ('FPP01' if m == 'fpp_lib.py' else 'FPF00') + ' first')
    shutil.copy2(src, Path('/content') / m)
sys.path.insert(0, '/content')
import fpf_lib as L; importlib.reload(L)
import fpf_tf as T; importlib.reload(T)
import fpp_lib as P; importlib.reload(P)
import numpy as np, pandas as pd, json, time, os, gc
if tuple(int(x) for x in P.VERSION.split('.')) < (1, 3): raise RuntimeError(f'fpp_lib {P.VERSION} in output/00_code is too old: upload fpp_lib 1.3 (validation threshold)')
ENV = L.env_info(); print('fpp_lib', P.VERSION, '|', json.dumps(ENV, indent=1))

## 1. Configuration

In [ ]:
import h5py, subprocess, math, tensorflow as tf
import matplotlib.pyplot as plt
KERNEL = 'FPP04L'; INIT_KERNEL = 'FPP04C'; ws = L.workspace(); KDIR = P.kernel_dir(ws, KERNEL); FIG = KDIR / 'figures'
REPS = L.repetitions(ws); REPS_TO_RUN = REPS
MODELS_ = ['EfficientNetV2S', 'MobileNetV3Large']; LOSSES = ['bce_control', 'bce_cosface', 'bce_proxyanchor']
BACKBONES = [f'{m}__{l}' for m in MODELS_ for l in LOSSES]          # one run name per (backbone, loss) arm
LAMBDA_PROXY = 1.0; COSFACE_S = 30.0; COSFACE_M = 0.35; PA_ALPHA = 32.0; PA_DELTA = 0.1; PROXY_NORM = 0.01; INIT_CHECK_MIN = 0.99
ARMS_TO_RUN = BACKBONES            # e.g. [a for a in BACKBONES if a.startswith('MobileNet')] to split the work across sessions
EMBEDDING_DIM = 1024; IMAGE_SIZE = 224; DROPOUT = 0.30
LR_FULL = 1e-5; WEIGHT_DECAY = 1e-4; EPOCHS = 50; PATIENCE = 10; LR_PATIENCE = 3; LR_FACTOR = 0.3
PAIR_BATCH = 64; TRAIN_PAIRS_PER_EPOCH = 12000; VAL_PAIRS = None; EMBED_BATCH = 256     # A100; VAL_PAIRS caps the validation pairs per set (smoke test only)
JIT_COMPILE = False   # XLA: often 10-30 % faster on A100; set True per session if every backbone compiles
VIT_PRESET = 'vit_base_patch16_224_imagenet'; PARTIAL_LAST_FRACTION_FALLBACK = 0.20
TIME_CPU = False; TIMING_REPEATS = 30
TRAIN_AUGMENTATION = True; AUG_P_ORIGINAL = 0.25     # training images: original with p 0.25, else one of the 3 FPP01 copies (uniform)
QUICK_SMOKE_TEST = False
if QUICK_SMOKE_TEST:
    EPOCHS = 1; TRAIN_PAIRS_PER_EPOCH = 384; VAL_PAIRS = 384; ARMS_TO_RUN = ['MobileNetV3Large__bce_cosface', 'MobileNetV3Large__bce_proxyanchor']; REPS_TO_RUN = REPS[:1]; KDIR = KDIR / '_smoke_test'; FIG = KDIR / 'figures'
FIG.mkdir(parents=True, exist_ok=True); (KDIR / 'runs').mkdir(parents=True, exist_ok=True); LOG = L.RunLog(KDIR / 'FPP04L_run_log.csv')
MIXED = T.use_mixed_precision()
CONFIG = dict(kernel=KERNEL, kernel_name=P.FPP_KERNELS[KERNEL], protocol=P.PROTOCOL_TABLE, arms=BACKBONES, models=MODELS_, losses=LOSSES, policy='full (all layers except BatchNorm), initialised from FPP04C', init_kernel=INIT_KERNEL, proxies='one per training finger (subject+finger), SHARED by DB1 and DB2, discarded after training', lambda_proxy=LAMBDA_PROXY, cosface=dict(s=COSFACE_S, m=COSFACE_M), proxy_anchor=dict(alpha=PA_ALPHA, delta=PA_DELTA), proxy_norm_parameterisation=PROXY_NORM, embedding_dim=EMBEDDING_DIM, image_size=IMAGE_SIZE, dropout=DROPOUT,
              loss='cosine-BCE (learned calibration)', lr=LR_FULL, weight_decay=WEIGHT_DECAY, epochs=EPOCHS, patience=PATIENCE, lr_patience=LR_PATIENCE, lr_factor=LR_FACTOR,
              pair_batch=PAIR_BATCH, train_pairs_per_epoch=TRAIN_PAIRS_PER_EPOCH, sampler='source-balanced DB1 / DB2 / DB1xDB2', validation='all DB1, DB2, DB1xDB2 validation pairs, embedded once per epoch', early_stopping=P.SELECTION_RULE,
              local_runs=str(P.LOCAL_RUNS), embed_batch=EMBED_BATCH, jit_compile=JIT_COMPILE,
              batchnorm='frozen', mixed_precision=MIXED, training_augmentation=(f'materialized FPP01 copies x{P.AUG_COPIES}, original with p {AUG_P_ORIGINAL}' if TRAIN_AUGMENTATION else 'none'),
              augmentation_recipe=P.AUG_RECIPE if TRAIN_AUGMENTATION else None, threshold=P.THRESHOLD_RULE, metrics=P.METRICS, ranking=f'mean {P.SELECTION_SET} EER over repetitions (AUC breaks ties)', vit_preset=VIT_PRESET,
              repetitions=REPS, quick_smoke_test=QUICK_SMOKE_TEST, fpp_lib_version=P.VERSION)
print(json.dumps(CONFIG, indent=2, default=str))

## 2. Images (FPP02), protocol and augmented copies (FPP01), shared finger identities

In [ ]:
IMG = {}
for d in L.DATASETS:
    with h5py.File(L.local_copy(ws['images'] / f'{d}_standardized_grayscale_{IMAGE_SIZE}.h5'), 'r') as h:
        if not h['complete'][:].all(): raise RuntimeError(f'{d}: image cache incomplete; finish FPF00')
        IMG[d] = h['images'][:]
    print(d, IMG[d].shape)
MOLF = ('MOLF_DB1', 'MOLF_DB2'); IMG_MOLF = np.concatenate([IMG[d] for d in MOLF]); OFFSET = {'MOLF_DB1': 0, 'MOLF_DB2': len(IMG['MOLF_DB1'])}
PROTO = {r: P.load_pairs(ws, r) for r in REPS}; print('FPP01 pair files verified for', len(REPS), 'repetitions')
AUG_MOLF = None
if TRAIN_AUGMENTATION:
    AUG = {}
    for d in MOLF:
        f = P.augmentation_file(ws, d)
        if not f.is_file(): raise FileNotFoundError(f'{f} missing: run FPP01 section 6 first')
        with h5py.File(L.local_copy(f), 'r') as h:
            if not h['complete'][:].all(): raise RuntimeError(f'{d}: augmentation incomplete; finish FPP01')
            if h.attrs['source_sha256'] != L.sha256_of(L.local_copy(ws['images'] / f'{d}_standardized_grayscale_{IMAGE_SIZE}.h5')): raise RuntimeError(f'{d}: augmented copies were made from a different image cache')
            AUG[d] = h['images'][:]
        print(d, 'augmented copies', AUG[d].shape)
    AUG_MOLF = np.concatenate([AUG[d] for d in MOLF]); del AUG                      # same index space as IMG_MOLF: (image, copy, H, W, 1)
class PairSeq(tf.keras.utils.Sequence):
    # Keras view of the fpp_lib source-balanced sampler; training: each image original (p AUG_P_ORIGINAL) or one of its stored copies, reproducible per (seed, epoch, batch)
    def __init__(self, images, sampler, aug=None, p_original=1.0, seed=0, workers=4):
        super().__init__(workers=workers, use_multiprocessing=False, max_queue_size=8); self.images = images; self.s = sampler; self.aug = aug; self.p = p_original; self.seed = seed; self.epoch = 0
    def __len__(self): return len(self.s)
    def __getitem__(self, b):
        ia, ib, y = self.s.batch(b)
        if self.aug is None: return (self.images[ia], self.images[ib]), y
        rng = np.random.default_rng([self.seed, self.epoch, b]); return (P.mix_versions(self.images, self.aug, ia, self.p, rng), P.mix_versions(self.images, self.aug, ib, self.p, rng)), y
    def on_epoch_end(self): self.s.on_epoch_end(); self.epoch += 1
# ---- shared cross-sensor identities: one class per TRAINING finger (subject + finger), the same class for its DB1 and DB2 images
def finger_classes(proto):
    lab = np.full(len(IMG_MOLF), -1, np.int64); keys = {}
    for d in MOLF:
        s = proto['samples'][d]; t_ = s[s.partition == 'train']
        # identity_id = subject|hand|finger, identical for the DB1 and DB2 captures of a finger (the same key FPP01 uses for the DB1xDB2 genuine pairs)
        for pos, key in zip(t_.sample_index.to_numpy(), t_.identity_id.astype(str).to_numpy()): lab[OFFSET[d] + int(pos)] = keys.setdefault(key, len(keys))
    sens = {k: set() for k in range(len(keys))}
    for d in MOLF:
        for k in np.unique(lab[OFFSET[d]:OFFSET[d] + len(IMG[d])]):
            if k >= 0: sens[int(k)].add(d)
    both = sum(len(v) == 2 for v in sens.values())
    if both != len(keys): raise RuntimeError(f'{len(keys) - both} training fingers lack one sensor: proxies would not be shared')
    return lab, len(keys)
CLASSES = {r: finger_classes(PROTO[r][0]) for r in REPS}
for r, (lab, k) in CLASSES.items():
    tr_idx = np.concatenate([np.asarray(PROTO[r][1][n]['index_a']) + OFFSET[P.PAIR_SETS[n][0]] for n in P.TRAIN_SOURCES] + [np.asarray(PROTO[r][1][n]['index_b']) + OFFSET[P.PAIR_SETS[n][1]] for n in P.TRAIN_SOURCES])
    if (lab[tr_idx] < 0).any(): raise RuntimeError(f'{r}: a training pair uses an image without a training finger class')
    print(r, '|', k, 'shared finger proxies (every one has DB1 and DB2 images) |', int((lab >= 0).sum()), 'labelled training images')

class ProxyPairSeq(PairSeq):
    # PairSeq + the shared finger class of each image of the pair (for the proxy losses)
    def __init__(self, labels, *a, **k): super().__init__(*a, **k); self.labels = labels
    def __getitem__(self, b):
        (xa, xb), y = super().__getitem__(b); ia, ib, _ = self.s.batch(b); return (xa, xb, self.labels[ia], self.labels[ib]), y

## 3. Encoder (FPM04 code, `full` policy), proxy Siamese model, balanced-EER callback

In [ ]:
A = tf.keras.applications
RULES = {'DenseNet201': ('prefix', ('conv5_',)), 'ResNet50': ('prefix', ('conv5_',)), 'VGG16': ('prefix', ('block5_',)), 'InceptionV3': ('after', 'mixed8'),
         'EfficientNetV2S': ('prefix', ('block6', 'top_')), 'MobileNetV3Large': ('prefix', ('expanded_conv_12', 'expanded_conv_13', 'expanded_conv_14', 'conv_1', 'Conv_1')),
         'ConvNeXtTiny': ('contains', ('stage_3', 'downsampling_block_2')), 'ViTB16': ('last_fraction', PARTIAL_LAST_FRACTION_FALLBACK)}

def make_backbone(name):
    # Returns (backbone model: (224,224,3) float 0..255 -> feature vector, preprocessing function or None)
    shape = (IMAGE_SIZE, IMAGE_SIZE, 3); kw = dict(include_top=False, weights='imagenet', input_shape=shape, pooling='avg')
    if name == 'DenseNet201': return A.DenseNet201(**kw), A.densenet.preprocess_input
    if name == 'ResNet50': return A.ResNet50(**kw), A.resnet50.preprocess_input
    if name == 'VGG16': return A.VGG16(**kw), A.vgg16.preprocess_input
    if name == 'InceptionV3': return A.InceptionV3(**kw), A.inception_v3.preprocess_input
    if name == 'EfficientNetV2S': return A.EfficientNetV2S(**kw), None                      # built-in preprocessing (expects 0..255)
    if name == 'MobileNetV3Large': return A.MobileNetV3Large(**kw), None                    # built-in preprocessing (expects 0..255)
    if name == 'ConvNeXtTiny': return A.ConvNeXtTiny(**kw), None                            # built-in preprocessing (expects 0..255)
    if name == 'ViTB16':
        try: import keras_hub
        except ImportError:                                                      # installed only when needed, without touching TensorFlow/Keras
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'keras-hub', 'kagglehub'], check=True); import keras_hub
        core = keras_hub.models.ViTBackbone.from_preset(VIT_PRESET)
        try: conv = keras_hub.layers.ViTImageConverter.from_preset(VIT_PRESET); pre = lambda z: conv(z)
        except Exception as exc: print('ViT image converter unavailable, using [-1, 1] scaling:', exc); pre = lambda z: z / 127.5 - 1.0
        inp = tf.keras.Input(shape); seq = core(pre(inp)); feat = tf.keras.layers.Lambda(lambda z: z[:, 0, :], name='cls_token')(seq)
        return tf.keras.Model(inp, feat, name='vit_b16_backbone'), None
    raise ValueError(name)

def leaf_layers(model):
    try: allv = list(model._flatten_layers(include_self=False, recursive=True))
    except AttributeError: allv = list(model.layers)
    out = []
    for l in allv:
        try: kids = list(l._flatten_layers(include_self=False, recursive=False))
        except AttributeError: kids = []
        if not kids: out.append(l)
    return out

def apply_policy(bb, name, policy):
    # 'full': everything trainable except BatchNorm. 'partial': only the layers named by RULES (BatchNorm always frozen).
    bb.trainable = True; leaves = leaf_layers(bb); rule, arg = RULES[name]; used = rule
    if policy == 'partial':
        if rule == 'prefix': sel = [l.name.startswith(arg) for l in leaves]
        elif rule == 'contains': sel = [any(a in l.name for a in arg) for l in leaves]
        elif rule == 'after':
            names = [l.name for l in leaves]; k = names.index(arg) if arg in names else len(names); sel = [i > k for i in range(len(names))]
        else: sel = [False] * len(leaves)
        weighted = [i for i, l in enumerate(leaves) if l.weights]
        if not any(sel[i] for i in weighted):
            used = f'last_fraction {PARTIAL_LAST_FRACTION_FALLBACK}' if rule == 'last_fraction' else f'fallback last_fraction {PARTIAL_LAST_FRACTION_FALLBACK} (rule matched nothing)'
            keep = set(weighted[int(len(weighted) * (1 - PARTIAL_LAST_FRACTION_FALLBACK)):]); sel = [i in keep for i in range(len(leaves))]
        for l, s in zip(leaves, sel): l.trainable = bool(s)
    for l in leaves:
        if isinstance(l, tf.keras.layers.BatchNormalization): l.trainable = False
    unfrozen = [l.name for l in leaves if l.trainable and l.weights]
    tr = int(sum(np.prod(w.shape) for w in bb.trainable_weights)); tot = int(sum(np.prod(w.shape) for w in bb.weights))
    return {'policy': policy, 'rule': used, 'unfrozen_layers': len(unfrozen), 'first_unfrozen': unfrozen[0] if unfrozen else '', 'last_unfrozen': unfrozen[-1] if unfrozen else '',
            'backbone_params_trainable': tr, 'backbone_params_total': tot}

def build_encoder(name, policy, emb_dim=EMBEDDING_DIM):
    bb, pre = make_backbone(name); info = apply_policy(bb, name, policy)
    image = tf.keras.Input((IMAGE_SIZE, IMAGE_SIZE, 1), dtype=tf.uint8, name='fingerprint_uint8')
    x = tf.keras.layers.Lambda(lambda z: tf.cast(z, tf.float32), name='cast_float')(image); x = tf.keras.layers.Concatenate(name='gray_to_rgb')([x, x, x])
    if pre is not None: x = tf.keras.layers.Lambda(pre, name='imagenet_preprocess')(x)
    f = bb(x, training=False); f = tf.keras.layers.Dropout(DROPOUT, name='head_dropout')(f)
    e = tf.keras.layers.Dense(emb_dim, use_bias=False, name='embedding_dense')(f); e = tf.keras.layers.BatchNormalization(name='embedding_bn')(e)
    e = tf.keras.layers.Lambda(lambda z: tf.math.l2_normalize(tf.cast(z, tf.float32), axis=1), name='embedding')(e)
    return tf.keras.Model(image, e, name=f'{name}_{policy}_encoder'), info

def build_siamese_model(encoder, lr):
    shape = (IMAGE_SIZE, IMAGE_SIZE, 1); ia = tf.keras.Input(shape, dtype=tf.uint8, name='input_a'); ib = tf.keras.Input(shape, dtype=tf.uint8, name='input_b')
    cos = tf.keras.layers.Dot(axes=1, name='cosine_similarity')([encoder(ia), encoder(ib)])
    model = tf.keras.Model([ia, ib], T.CosineCalibration(name='same_finger_probability')(cos), name='siamese')
    model.compile(optimizer=tf.keras.optimizers.AdamW(learning_rate=lr, weight_decay=WEIGHT_DECAY), loss=tf.keras.losses.BinaryCrossentropy(),
                  metrics=[tf.keras.metrics.BinaryAccuracy(name='verification_accuracy'), tf.keras.metrics.AUC(name='verification_auc')], jit_compile=JIT_COMPILE)
    return model
class EpochClock(tf.keras.callbacks.Callback):
    def on_epoch_begin(self, epoch, logs=None): self.t0 = time.perf_counter()
    def on_epoch_end(self, epoch, logs=None):
        if logs is not None: logs['epoch_seconds'] = time.perf_counter() - self.t0
class BalancedEERCallback(tf.keras.callbacks.Callback):
    # after each epoch: embed the validation images, score all validation pairs, source-balanced EER -> best checkpoint / LR drop / early stop (fpp_lib rule)
    def __init__(self, encoder, bv, ckpt, hist_csv):
        super().__init__(); self.encoder = encoder; self.bv = bv; self.ckpt = ckpt; self.hist_csv = hist_csv; self.x = IMG_MOLF[bv.indices]; self.rows = []
        self.stop = P.BalancedEERStopping(PATIENCE, LR_PATIENCE, LR_FACTOR)
    def _lr(self): return float(np.asarray(self.model.optimizer.learning_rate))
    def _set_lr(self, v):
        try: self.model.optimizer.learning_rate.assign(v)
        except AttributeError: self.model.optimizer.learning_rate = v
    def on_epoch_end(self, epoch, logs=None):
        logs = logs if logs is not None else {}; t0 = time.perf_counter()
        m = self.bv.score(L.l2n(T.embed(self.encoder, self.x, EMBED_BATCH))); logs.update(m); lr = self._lr(); d = self.stop.update(epoch, m['val_balanced_eer'], m['val_balanced_auc'])
        if d['improved']: self.model.save_weights(str(self.ckpt))
        if d['reduce_lr'] and lr > self.stop.min_lr: self._set_lr(max(lr * LR_FACTOR, self.stop.min_lr))
        if d['stop']: self.model.stop_training = True
        logs['validation_seconds'] = time.perf_counter() - t0
        self.rows.append({'epoch': epoch + 1, **{k: float(np.asarray(v)) for k, v in logs.items() if np.asarray(v).size == 1}, 'lr': lr, 'improved': d['improved']}); pd.DataFrame(self.rows).to_csv(self.hist_csv, index=False)
        print(f"  epoch {epoch + 1}: val balanced EER {100 * m['val_balanced_eer']:.2f} % | AUC {100 * m['val_balanced_auc']:.2f} % | "
              + ' '.join(f"{n.split('_')[0]} {100 * m[f'val_eer_{n}']:.2f}" for n in P.VAL_SOURCES) + (' *best' if d['improved'] else '') + (' | LR down' if d['reduce_lr'] else '') + (' | stop' if d['stop'] else ''))
class ProxySiamese(tf.keras.Model):
    # shared encoder; loss = cosine-BCE on the pair  +  LAMBDA_PROXY * proxy loss on all 2B images against the SHARED finger proxies (none for the control)
    def __init__(self, encoder, n_classes, kind, init_proxies, **kw):
        super().__init__(**kw); self.encoder = encoder; self.kind = kind; self.calib = T.CosineCalibration(name='same_finger_probability')
        self.proxies = self.add_weight(name='finger_proxies', shape=(n_classes, EMBEDDING_DIM), dtype='float32', trainable=kind != 'bce_control', initializer='zeros')
        self.proxies.assign(np.asarray(init_proxies, np.float32) * PROXY_NORM)          # finger mean embeddings, small norm (faster angular learning)
        self.n_classes = n_classes; self.bce = tf.keras.losses.BinaryCrossentropy()
        self.t_loss = tf.keras.metrics.Mean(name='loss'); self.t_bce = tf.keras.metrics.Mean(name='bce'); self.t_proxy = tf.keras.metrics.Mean(name='proxy_loss')
        self.t_acc = tf.keras.metrics.BinaryAccuracy(name='verification_accuracy'); self.t_auc = tf.keras.metrics.AUC(name='verification_auc')
    @property
    def metrics(self): return [self.t_loss, self.t_bce, self.t_proxy, self.t_acc, self.t_auc]
    def call(self, inputs, training=False):
        xa, xb = inputs[0], inputs[1]; ea = self.encoder(xa, training=training); eb = self.encoder(xb, training=training)
        return self.calib(tf.reduce_sum(tf.cast(ea, tf.float32) * tf.cast(eb, tf.float32), axis=1))
    def proxy_loss(self, E, lab):
        W = tf.math.l2_normalize(self.proxies, axis=1); cos = tf.matmul(E, W, transpose_b=True); onehot = tf.one_hot(lab, self.n_classes, dtype=tf.float32)
        if self.kind == 'bce_cosface':
            logits = COSFACE_S * (cos - COSFACE_M * onehot); return tf.reduce_mean(tf.nn.softmax_cross_entropy_with_logits(onehot, logits))
        # Proxy-Anchor (Kim et al. 2020): positives pulled to their proxy, negatives pushed, hardest weighted most
        pos = tf.exp(-PA_ALPHA * (cos - PA_DELTA)) * onehot; neg = tf.exp(PA_ALPHA * (cos + PA_DELTA)) * (1.0 - onehot)
        with_pos = tf.cast(tf.reduce_sum(onehot, 0) > 0, tf.float32)
        lp = tf.reduce_sum(tf.math.log1p(tf.reduce_sum(pos, 0)) * with_pos) / tf.maximum(tf.reduce_sum(with_pos), 1.0)
        ln = tf.reduce_sum(tf.math.log1p(tf.reduce_sum(neg, 0))) / float(self.n_classes); return lp + ln
    def train_step(self, data):
        (xa, xb, la, lb), y = data; y = tf.cast(tf.reshape(y, [-1]), tf.float32)
        with tf.GradientTape() as tape:
            ea = tf.cast(self.encoder(xa, training=True), tf.float32); eb = tf.cast(self.encoder(xb, training=True), tf.float32)
            p = self.calib(tf.reduce_sum(ea * eb, axis=1)); l_bce = self.bce(y, p)
            l_px = tf.constant(0.0) if self.kind == 'bce_control' else self.proxy_loss(tf.concat([ea, eb], 0), tf.cast(tf.concat([la, lb], 0), tf.int32))
            loss = l_bce + LAMBDA_PROXY * l_px
            scaled = self.optimizer.scale_loss(loss) if hasattr(self.optimizer, 'scale_loss') else loss
        tv = self.trainable_variables; self.optimizer.apply_gradients(zip(tape.gradient(scaled, tv), tv))
        self.t_loss.update_state(loss); self.t_bce.update_state(l_bce); self.t_proxy.update_state(l_px); self.t_acc.update_state(y, p); self.t_auc.update_state(y, p)
        return {m.name: m.result() for m in self.metrics}

def build_proxy_model(encoder, n_classes, kind, init_proxies):
    m = ProxySiamese(encoder, n_classes, kind, init_proxies, name='proxy_siamese')
    dummy = np.zeros((1, IMAGE_SIZE, IMAGE_SIZE, 1), np.uint8); m((dummy, dummy)); m.built = True     # build once so checkpoints can be saved (Keras 3)
    m.compile(optimizer=tf.keras.optimizers.AdamW(learning_rate=LR_FULL, weight_decay=WEIGHT_DECAY), jit_compile=JIT_COMPILE); return m
print('Arms ready:', BACKBONES)

## 4. One run = one (backbone, loss, repetition), initialised from FPP04C; proxies at the finger mean embeddings

In [ ]:
def cpu_latency(encoder, backbone):
    # single-image latency on the CPU with an identical copy of the trained encoder (float32)
    try:
        w = encoder.get_weights(); pol = tf.keras.mixed_precision.global_policy().name; tf.keras.mixed_precision.set_global_policy('float32')
        with tf.device('/CPU:0'):
            ec, _ = build_encoder(backbone, 'partial'); ec.set_weights(w); _, ms = T.time_encoder(ec, IMG['MOLF_DB1'][:4], TIMING_REPEATS)
        tf.keras.mixed_precision.set_global_policy(pol); del ec; gc.collect(); return ms
    except Exception as exc:
        LOG.add(step='cpu_timing_failed', arm=backbone, status='warning', error=repr(exc)[:300]); tf.keras.mixed_precision.set_global_policy('mixed_float16' if MIXED else 'float32'); return float('nan')

def load_init(encoder, model_name, rep):
    src = Path(ws['output']) / P.FPP_KERNELS[INIT_KERNEL] / 'runs' / rep / model_name
    if not L.run_complete(src): raise FileNotFoundError(f'{src} not finished: run FPP04C first')
    encoder.load_weights(str(src / 'encoder.weights.h5'))
    E0 = L.load_npz(src / 'embeddings_float32.npz')['MOLF_DB1'][:32]; E1 = L.l2n(T.embed(encoder, IMG['MOLF_DB1'][:32], 32)); cs = float(np.mean(np.sum(L.l2n(E0) * E1, axis=1)))
    if cs < INIT_CHECK_MIN: raise RuntimeError(f'{model_name} {rep}: reloaded FPP04C encoder does not reproduce its embeddings (mean cosine {cs:.4f})')
    return {'init_from': L.rel(ws, src), 'init_check_cosine': round(cs, 6)}

def init_proxies(encoder, labels, n_classes):
    # mean L2-normalised embedding of each shared finger class (DB1 and DB2 training originals together), L2-normalised
    idx = np.flatnonzero(labels >= 0); E = L.l2n(T.embed(encoder, IMG_MOLF[idx], EMBED_BATCH)); W = np.zeros((n_classes, EMBEDDING_DIM), np.float32)
    np.add.at(W, labels[idx], E); return L.l2n(W)

def rescore_run(backbone, rep, rd, done):
    # finished run evaluated under an older threshold rule: same model, same saved pair scores, new frozen threshold (validation EER)
    S = P.load_scores(rd); old = json.loads((rd / 'thresholds.json').read_text())
    extra = {k: v for k, v in done.items() if k not in ('kernel', 'arm', 'repetition', 'seed', 'completed_utc', 'threshold_rule') and not k.startswith('threshold_')}
    extra['rescored_from'] = json.dumps({k: v for k, v in old.items() if k != 'rule'})
    rows, thr = P.evaluate(S, KERNEL, backbone, rep, extra); P.save_run(rd, S, rows, thr, {'kernel': KERNEL, 'arm': backbone, 'repetition': rep, 'seed': done.get('seed'), **extra})
    P.fig_set_panel(f'{backbone} {rep[:6]} (frozen validation-EER threshold)', {st: [(*P.group_scores(S, st), thr)] for st in P.PAPER_SETS}, FIG / 'runs' / rep / f'{backbone}_panel.png')
    v = [r_ for r_ in rows if r_['set'] == 'MOLF test (balanced)'][0]
    LOG.add(step='rescored', arm=backbone, repetition=rep, status='ok', threshold_old=round(float(next(iter(old.values()))), 4), threshold_validation=round(thr, 4),
            test_balanced_f1=round(100 * v['f1_macro'], 2), test_balanced_fmr=round(100 * v['fmr'], 2), test_balanced_fnmr=round(100 * v['fnmr'], 2))

def train_run(backbone, rep):
    rd = KDIR / 'runs' / rep / backbone
    if L.run_complete(rd):
        done = json.loads((rd / 'run_complete.json').read_text())
        if done.get('early_stopping') == P.SELECTION_RULE:
            if P.threshold_current(rd): print(backbone, rep, 'complete -> skipped'); return
            rescore_run(backbone, rep, rd, done); return
        old = rd.with_name(rd.name + '__old_stopping_rule'); rd.rename(old); print(backbone, rep, 'finished with an older stopping rule -> moved to', old.name, 'and retrained')
    lrd = P.local_run_dir(KERNEL + ('_smoke' if QUICK_SMOKE_TEST else ''), rep, backbone)          # Colab disk; copied to Drive once at the end
    model_name, loss_kind = backbone.split('__'); proto, Pp = PROTO[rep]; seed = proto['seed']; tf.keras.backend.clear_session(); gc.collect(); T.set_all_seeds(seed); t = L.timer()
    encoder, info = build_encoder(model_name, 'full'); init = load_init(encoder, model_name, rep); params = T.count_params(encoder)
    labels, n_cls = CLASSES[rep]; W0 = init_proxies(encoder, labels, n_cls); siamese = build_proxy_model(encoder, n_cls, loss_kind, W0)
    init.update(proxies=int(n_cls), proxies_shared_across_sensors=True, proxy_loss=loss_kind)
    ssf = float(proto['manifest']['same_subject_impostor_fraction'])
    train_seq = ProxyPairSeq(labels, IMG_MOLF, P.MixedPairs(P.sources(Pp, P.TRAIN_SOURCES, OFFSET), PAIR_BATCH, True, seed, TRAIN_PAIRS_PER_EPOCH, ssf), AUG_MOLF if TRAIN_AUGMENTATION else None, AUG_P_ORIGINAL, seed)
    bv = P.BalancedValidation(Pp, OFFSET, max_pairs=VAL_PAIRS)
    LOG.add(step='train_start', arm=backbone, repetition=rep, **init, train_composition=json.dumps(train_seq.s.composition), validation_pairs=json.dumps(bv.composition), validation_images=int(len(bv.indices)), **params, **info)
    ckpt = lrd / 'best_siamese.weights.h5'; hist_csv = lrd / 'training_history.csv'; fit_marker = lrd / 'fit_complete.json'
    if ckpt.is_file() and fit_marker.is_file():
        train_s = json.loads(fit_marker.read_text())['training_seconds']; print(backbone, rep, 'fit finished earlier in this session -> evaluating the local checkpoint')
    else:
        cbs = [EpochClock(), BalancedEERCallback(encoder, bv, ckpt, hist_csv)]
        siamese.fit(train_seq, epochs=EPOCHS, callbacks=cbs, verbose=2); train_s = t(); fit_marker.write_text(json.dumps({'training_seconds': train_s}))
    siamese.load_weights(ckpt)
    hist = pd.read_csv(hist_csv); b_i, _ = P.BalancedEERStopping.replay(hist, EPOCHS, patience=PATIENCE, lr_patience=LR_PATIENCE, lr_factor=LR_FACTOR); best = hist.iloc[b_i]
    emb = {d: L.l2n(T.embed(encoder, IMG[d], EMBED_BATCH)) for d in L.DATASETS}                         # clean images, inference mode
    encoder.save_weights(lrd / 'encoder.weights.h5'); np.savez_compressed(lrd / 'embeddings_float32.npz', **{d: e.astype(np.float32) for d, e in emb.items()})
    gpu_batch_ms, gpu_single_ms = T.time_encoder(encoder, IMG['MOLF_DB1'][:64], TIMING_REPEATS); cpu_single_ms = cpu_latency(encoder, model_name) if TIME_CPU else float('nan')
    extra = {'backbone': model_name, 'loss': loss_kind, 'policy': 'full', **init, 'training_augmentation': CONFIG['training_augmentation'], 'early_stopping': P.SELECTION_RULE, 'best_epoch': int(best.epoch), 'epochs_run': int(len(hist)),
             'val_balanced_eer_at_best': float(best.val_balanced_eer), 'val_balanced_auc_at_best': float(best.val_balanced_auc), 'train_auc_at_best': float(best.get('verification_auc', np.nan)),
             'training_seconds': train_s, 'seconds_per_epoch': float(hist['epoch_seconds'].mean()) if 'epoch_seconds' in hist else float('nan'),
             'validation_seconds_per_epoch': float(hist['validation_seconds'].mean()) if 'validation_seconds' in hist else float('nan'), 'pair_batch': PAIR_BATCH,
             'embed_ms_per_image_gpu_batch64': gpu_batch_ms, 'embed_ms_single_image_gpu': gpu_single_ms, 'embed_ms_single_image_cpu': cpu_single_ms, **params, **info}
    S = {n: (np.asarray(Pp[n]['label'], int), L.pair_scores(emb[da], emb[db], Pp[n]).astype(np.float64)) for n, (da, db, _) in P.PAIR_SETS.items()}
    rows, thr = P.evaluate(S, KERNEL, backbone, rep, extra)
    (lrd / 'encoder_config.json').write_text(json.dumps({'backbone': model_name, 'loss': loss_kind, 'policy': 'full', 'init_from': init['init_from'], 'rules': RULES[model_name], 'embedding_dim': EMBEDDING_DIM, 'image_size': IMAGE_SIZE,
                                                         'dropout': DROPOUT, 'learning_rate': LR_FULL, 'proxies': 'discarded after training (not part of the deployed encoder)', **info}, indent=2, default=str))
    P.save_run(lrd, S, rows, thr, {'kernel': KERNEL, 'arm': backbone, 'repetition': rep, 'seed': seed, **extra})
    n_files = P.publish_run(lrd, rd, skip=('fit_complete.json', 'best_siamese.weights.h5'))   # the training checkpoint holds the proxies: not kept; tc = L.timer()                     # one copy to Drive, run_complete.json last
    P.fig_set_panel(f'{backbone} {rep[:6]} (frozen validation-EER threshold)', {st: [(*P.group_scores(S, st), thr)] for st in P.PAPER_SETS}, FIG / 'runs' / rep / f'{backbone}_panel.png')
    P.fig_training_balanced(hist, f'{backbone} {rep[:6]} — stopping on source-balanced validation EER', FIG / 'runs' / rep / f'{backbone}_training.png')
    v = [r_ for r_ in rows if r_['set'] == P.SELECTION_SET][0]
    LOG.add(step='train_done', arm=backbone, repetition=rep, status='ok', minutes=round(t() / 60, 1), best_epoch=int(best.epoch), files_to_drive=n_files,
            val_balanced_eer=round(100 * v['eer'], 2), val_balanced_auc=round(100 * v['roc_auc'], 2), val_balanced_f1=round(100 * v['f1_macro'], 2))
    del siamese, encoder, emb; gc.collect()

## 5. Train every arm for every repetition

In [ ]:
for b in ARMS_TO_RUN:
    for rep in REPS_TO_RUN:
        try: train_run(b, rep)
        except (tf.errors.ResourceExhaustedError, tf.errors.InternalError) as exc:
            LOG.add(step='train_failed', arm=b, repetition=rep, status='gpu_error', error=repr(exc)[:400]); print('GPU error', b, rep, repr(exc)[:200]); tf.keras.backend.clear_session(); gc.collect()
        except Exception as exc:
            LOG.add(step='train_failed', arm=b, repetition=rep, status='error', error=repr(exc)[:400]); print('FAILED', b, rep, exc)

## 6. Results, ranking (validation only); paired vs `bce_control` and vs FPP04C

In [ ]:
RES, HIST, MISSING = [], [], []
for b in BACKBONES:
    for r in REPS_TO_RUN:
        rd = KDIR / 'runs' / r / b
        if not L.run_complete(rd): MISSING.append((b, r)); continue
        RES.append(pd.read_csv(rd / 'metrics.csv')); h = pd.read_csv(rd / 'training_history.csv').drop(columns='epoch', errors='ignore'); h.insert(0, 'epoch', np.arange(1, len(h) + 1)); HIST.append(h.assign(arm=b, repetition=r))
if MISSING: print('Not finished yet:', MISSING)
if not RES: raise RuntimeError('no finished run: run section 5 first')
results = pd.concat(RES, ignore_index=True); results['kernel'] = results['arm']; histories = pd.concat(HIST, ignore_index=True)
complete = [b for b in BACKBONES if all(L.run_complete(KDIR / 'runs' / r / b) for r in REPS_TO_RUN)]
ranking = P.selection_table(results, complete); WINNER = ranking.arm.iloc[0]
first = results[results.set == P.SELECTION_SET]
TCOLS = ['parameters_total', 'parameters_trainable', 'embed_ms_per_image_gpu_batch64', 'embed_ms_single_image_gpu', 'embed_ms_single_image_cpu', 'training_seconds', 'seconds_per_epoch',
         'validation_seconds_per_epoch', 'best_epoch', 'epochs_run']
timing = first.groupby('arm')[[c for c in TCOLS if c in first]].agg(['mean', 'std']); timing.columns = [f'{a}_{b}' for a, b in timing.columns]; timing = timing.reset_index()
ranking = ranking.merge(timing[['arm'] + [c for c in timing.columns if c.endswith('_mean')]], on='arm', how='left')
best_eer = ranking.val_eer_mean.min(); ranking['val_eer_gap_to_best'] = ranking.val_eer_mean - best_eer
ranking['within_0.5_EER_of_best'] = ranking.val_eer_gap_to_best <= 0.5
print('Ranking by source-balanced validation EER (AUC breaks ties):'); display(ranking.round(3))
paper, snum = P.paper_table(results, complete); wide = P.wide_table(snum, complete); display(wide)
paired = P.paired_differences(results, WINNER, [b for b in complete if b != WINNER])
overfit = first[['arm', 'repetition', 'best_epoch', 'epochs_run', 'train_auc_at_best', 'val_balanced_auc_at_best', 'val_balanced_eer_at_best']].copy()
overfit['auc_gap_points'] = 100 * (overfit.train_auc_at_best - overfit.val_balanced_auc_at_best)   # train AUC is on augmented batches
# ---- vs bce_control (proxy effect) and vs FPP04C (the starting model, re-scored under the same threshold rule)
INIT_DIR = Path(ws['output']) / P.FPP_KERNELS[INIT_KERNEL]; REF = []
for m in MODELS_:
    rows_ = P.reference_rows(f'{m}__FPP04C', INIT_DIR, arm=m, reps=REPS_TO_RUN)
    for r_ in rows_: r_['arm'] = f'{m}__FPP04C'
    REF += rows_
ALLR = pd.concat([results] + ([pd.DataFrame(REF).assign(kernel=lambda x: x.arm)] if REF else []), ignore_index=True)
ORDER = [a for m in MODELS_ for a in [f'{m}__FPP04C'] + [f'{m}__{l}' for l in LOSSES] if a in set(ALLR.kernel)]
paper_l, snum_l = P.paper_table(ALLR, ORDER, sets=['MOLF validation (balanced)', 'MOLF test (balanced)', 'DB1xDB2_test', 'FingerDOS_test'])
wide_l = P.wide_table(snum_l, ORDER, sets=['MOLF test (balanced)', 'DB1xDB2_test', 'FingerDOS_test'], metrics_=('roc_auc', 'eer', 'fmr', 'fnmr', 'f1_macro'))
print('All arms vs the FPP04C start (frozen validation-EER threshold, mean ± std over repetitions):'); display(wide_l)
PAIRED = []
for m in MODELS_:
    for l in ('bce_cosface', 'bce_proxyanchor'):
        a = f'{m}__{l}'
        if a in set(ALLR.kernel): PAIRED.append(P.paired_differences(ALLR, a, [x for x in (f'{m}__bce_control', f'{m}__FPP04C') if x in set(ALLR.kernel)]))
    if f'{m}__bce_control' in set(ALLR.kernel) and f'{m}__FPP04C' in set(ALLR.kernel): PAIRED.append(P.paired_differences(ALLR, f'{m}__bce_control', [f'{m}__FPP04C']))
paired_l = pd.concat(PAIRED, ignore_index=True) if PAIRED else pd.DataFrame()
if len(paired_l):
    print('Paired differences (points; negative EER = better):')
    display(paired_l[paired_l.set.isin(['MOLF test (balanced)', 'DB1xDB2_test', 'FingerDOS_test']) & paired_l.metric.isin(['eer', 'f1_macro'])].round(3))

## 6b. Bias check of the threshold: 5 calibration hold-out subjects vs validation (MOLF test, balanced)

In [ ]:
CB = []
for b in complete:
    for r in REPS_TO_RUN:
        rd = KDIR / 'runs' / r / b; proto, Pp = PROTO[r]
        c = P.calibration_check(P.load_scores(rd), L.load_npz(rd / 'embeddings_float32.npz'), P.calibration_pairs(proto)); CB.append({'arm': b, 'repetition': r, **c})
calib = pd.DataFrame(CB); cs = calib.drop(columns='repetition').groupby('arm').agg(['mean', 'std'])
calib_table = pd.DataFrame({'backbone': cs.index,
    'threshold validation': [f"{a:.3f} ± {b_:.3f}" for a, b_ in zip(cs[('threshold_validation', 'mean')], cs[('threshold_validation', 'std')])],
    'threshold calibration (5 subj.)': [f"{a:.3f} ± {b_:.3f}" for a, b_ in zip(cs[('threshold_calibration', 'mean')], cs[('threshold_calibration', 'std')])],
    **{f'test {m.upper()} {tag} (%)': (100 * cs[(f'test_{m}_{tag}', 'mean')]).round(2).to_numpy() for m in ('fmr', 'fnmr', 'f1') for tag in ('validation', 'calibration')}}).reset_index(drop=True)
print('Bias check (paper: supplementary): same operating point from 5 never-used subjects -> the validation threshold is not biased by reuse for selection'); display(calib_table)

## 7. Figures

In [ ]:
MR = {**{b: [KDIR / 'runs' / r / b for r in REPS_TO_RUN] for b in complete}, **{f'{m}__FPP04C': [INIT_DIR / 'runs' / r / m for r in REPS_TO_RUN] for m in MODELS_ if f'{m}__FPP04C' in ORDER}}
P.standard_figures(FIG, MR, sets=P.PAPER_SETS)
P.fig_metric_bars(snum_l, ORDER, FIG / 'metric_bars_losses.png', sets=['MOLF test (balanced)', 'DB1xDB2_test', 'FingerDOS_test'])
for xcol, xl, tag in (('embed_ms_single_image_cpu_mean', 'CPU ms per image (single)', 'cpu'), ('embed_ms_per_image_gpu_batch64_mean', 'GPU ms per image (batch 64)', 'gpu'),
                      ('parameters_total_mean', 'parameters', 'params'), ('training_seconds_mean', 'training seconds per run', 'train_time')):
    if xcol in ranking and ranking[xcol].notna().any():
        P.fig_speed_accuracy(ranking.dropna(subset=[xcol]), xcol, 'val_eer_mean', 'arm', FIG / f'speed_accuracy_{tag}.png', f'Validation EER (balanced) vs {xl}', xl, f'{P.SELECTION_SET} EER (%)')
fig, ax = plt.subplots(figsize=(8, 4.5))
for k, b in enumerate(complete):
    h = histories[histories.arm == b].groupby('epoch').val_balanced_eer.mean(); ax.plot(h.index, 100 * h.values, color=P.COLORS[k % 10], label=b)
ax.set(xlabel='Epoch', ylabel='validation EER, source-balanced (%)', title='Training curves (mean over repetitions)'); ax.legend(fontsize=7); ax.grid(alpha=.3); P._save(fig, FIG / 'training_curves_backbones.png')

## 8. Manifest and workbook

In [ ]:
if not QUICK_SMOKE_TEST and len(complete) == len(BACKBONES):
    files = ('encoder.weights.h5', 'encoder_config.json', 'embeddings_float32.npz', 'thresholds.json', 'pair_scores.npz')
    WINNERS = {m: P.selection_table(results, [a for a in complete if a.startswith(m + '__')]).arm.iloc[0] for m in MODELS_}   # best loss per backbone
    per_rep = {r: {'seed': L.rep_seed(r), 'run_dir': L.rel(ws, KDIR / 'runs' / r / WINNER), 'files': {f: L.rel(ws, KDIR / 'runs' / r / WINNER / f) for f in files}} for r in REPS_TO_RUN}
    (KDIR / 'winner_manifest.json').write_text(json.dumps({'kernel': KERNEL, 'winning_arm': WINNER, 'winner_per_backbone': WINNERS, 'ranking': ranking.to_dict('records'), 'config': CONFIG,
                                                           'selection_rule': P.SELECTION_RULE, 'note': 'shared cross-sensor finger proxies (discarded after training); lowest source-balanced validation EER overall and per backbone',
                                                           'repetitions': per_rep}, indent=2, default=str))
SHEETS = {'Losses vs FPP04C (wide)': wide_l, 'Losses vs FPP04C': paper_l, 'Paired differences': paired_l, 'Ranking (validation)': ranking, 'Paper Table (wide)': wide, 'Paper Table': paper, 'Paired vs Best': paired, 'Time and Complexity': timing, 'Summary (numeric)': snum,
          'Calibration Bias Check': calib_table, 'Calibration (per rep)': calib, 'Verification Results': P.to_percent(results), 'Training Log': histories, 'Overfitting': overfit, 'Configuration': CONFIG, 'Run Log': LOG.frame(), 'Environment': ENV,
          'Figure Index': L.figure_index(KDIR)}
L.write_workbook(KDIR / 'FPP04L_Shared_Proxy_Loss_Results.xlsx', SHEETS)
for name, df in (('ranking', ranking), ('paper_table', paper), ('paper_table_wide', wide), ('losses_vs_fpp04c_wide', wide_l), ('paired_differences', paired_l), ('time', timing), ('results_per_repetition', P.to_percent(results)), ('calibration_bias_check', calib_table)): df.to_csv(KDIR / f'FPP04L_{name}.csv', index=False)
print('FPP04L complete | lowest source-balanced validation EER:', WINNER)